# 🎨 Colab → Hermes: генератор картинок (ComfyUI + SDXL)

Поднимает на видеокарте Google Colab **ComfyUI** — самый мощный открытый генератор картинок (модель SDXL класса Juggernaut), делает публичную ссылку, сам скачивает модель и **рисует тестовую картинку** для проверки. Потом этим генератором можно пользоваться прямо из Hermes: «сгенерируй картинку …».

**Как запустить:**
1. Меню **Среда выполнения → Сменить тип среды выполнения → аппаратный ускоритель: T4 GPU** → Сохранить.
2. Меню **Среда выполнения → Выполнить все** (Ctrl+F9). Предупреждение «ноутбук не от Google» → **Run anyway**.
3. Первый запуск: 6–12 минут (качается модель ~7 ГБ). В конце появится тестовая картинка и блок **«СКОПИРУЙ ЭТО…»** — пришли его в чат Hermes.

**Что важно:**
- Нужна видеокарта (T4 бесплатно). Без GPU не поедет.
- Сессия живёт до ~12 часов. Умерла — «Выполнить все» заново, ссылка новая.
- Одновременно один GPU-ноутбук (этот ИЛИ LLM-сервер). Оба сразу — только на платных GPU.

**Чекпоинты (меняются в ячейке 1):** `juggernaut` (универсальный фото-стиль, по умолчанию), `realvis` (фотореализм), `sdxl_base` (базовый SDXL).

In [ ]:
# ==== НАСТРОЙКИ ====
CHECKPOINT = "juggernaut"  # "juggernaut" | "realvis" | "sdxl_base"
PORT = 8188                # локальный порт ComfyUI
TEST_IMAGE = True          # сгенерировать тестовую картинку в конце (рекомендую оставить True)

CHECKPOINTS = {
    "juggernaut": ("RunDiffusion/Juggernaut-XL-v9", "Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"),
    "realvis":    ("SG161222/RealVisXL_V4.0",       "RealVisXL_V4.0.safetensors"),
    "sdxl_base":  ("stabilityai/stable-diffusion-xl-base-1.0", "sd_xl_base_1.0.safetensors"),
}
UPSCALER = ("lokCX/4x-Ultrasharp", "4x-UltraSharp.pth")
print("Настройки загружены. Чекпоинт:", CHECKPOINT)

In [ ]:
import subprocess, re

gpu_info = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_info if gpu_info else "НЕ НАЙДЕН")
if not gpu_info:
    raise RuntimeError(
        "Нет GPU. Сделай: Среда выполнения → Сменить тип среды выполнения → T4 GPU → Сохранить, "
        "затем: Среда выполнения → Выполнить все."
    )

In [ ]:
import subprocess, sys, os

os.makedirs("/content/logs", exist_ok=True)
if not os.path.isdir("/content/ComfyUI"):
    print("Клонирую ComfyUI...")
    subprocess.run("git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI", shell=True, check=True)

print("Ставлю зависимости (2–3 минуты)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_transfer]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=False)

if not os.path.isdir("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    subprocess.run("git clone -q https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "/content/ComfyUI/custom_nodes/ComfyUI-Manager/requirements.txt"], check=False)

print("✅ ComfyUI готов")

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

repo, fname = CHECKPOINTS[CHECKPOINT]
print("Качаю чекпоинт %s (~7 ГБ, 2–4 минуты)..." % fname)
p = hf_hub_download(repo_id=repo, filename=fname, local_dir="/content/ComfyUI/models/checkpoints")
print("✅ %s — %.1f ГБ" % (p, os.path.getsize(p) / 2**30))
CKPT_FILE = fname

u_repo, u_file = UPSCALER
p2 = hf_hub_download(repo_id=u_repo, filename=u_file, local_dir="/content/ComfyUI/models/upscale_models")
print("✅ %s — %.0f МБ" % (p2, os.path.getsize(p2) / 2**20))

In [ ]:
import subprocess, sys, time, json, urllib.request

LOG = "/content/logs/comfyui.log"
with open(LOG, "w") as lf:
    proc = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch"],
        cwd="/content/ComfyUI", stdout=lf, stderr=subprocess.STDOUT,
    )

ok = False
for _ in range(120):
    try:
        json.load(urllib.request.urlopen("http://127.0.0.1:%d/system_stats" % PORT, timeout=3))
        ok = True
        break
    except Exception:
        if proc.poll() is not None:
            break
        time.sleep(2)

if ok:
    print("✅ ComfyUI запущен локально на порту", PORT)
else:
    print("❌ Не поднялся. Лог:")
    print(open(LOG, errors="ignore").read()[-2000:])
    raise RuntimeError("ComfyUI не запустился — Перезапусти сессию и Выполнить все заново.")

In [ ]:
import subprocess, os, re, time, json, uuid, urllib.request, urllib.parse

# туннель
if not os.path.exists("/content/cloudflared"):
    subprocess.run(
        "wget -q -O /content/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "&& chmod +x /content/cloudflared",
        shell=True, check=True,
    )
TUN_LOG = "/content/logs/cloudflared.log"
with open(TUN_LOG, "w") as lf:
    subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:%d" % PORT, "--no-autoupdate"],
        stdout=lf, stderr=subprocess.STDOUT,
    )
url = None
for _ in range(60):
    mm = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUN_LOG, errors="ignore").read())
    if mm:
        url = mm.group(0)
        break
    time.sleep(2)
if not url:
    raise RuntimeError("Туннель не поднялся — запусти эту ячейку ещё раз")

print("Публичный адрес:", url)
print("Проверяю ComfyUI через туннель...")
urllib.request.urlopen(url + "/system_stats", timeout=60).read()
print("✅ сервер отвечает через туннель")

if TEST_IMAGE:
    wf = {
        "4": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": CKPT_FILE}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {"width": 1024, "height": 1024, "batch_size": 1}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["4", 1], "text": "cinematic photo of a fluffy ginger cat barista serving coffee to a corgi in a cozy cafe, warm window light, shallow depth of field, high detail"}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["4", 1], "text": "watermark, text, logo, blurry, low quality, deformed, extra limbs"}},
        "3": {"class_type": "KSampler", "inputs": {"seed": 777, "steps": 28, "cfg": 6.0, "sampler_name": "dpmpp_2m", "scheduler": "karras", "denoise": 1.0, "model": ["4", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["4", 2]}},
        "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": "hermes_test", "images": ["8", 0]}},
    }
    req = urllib.request.Request(
        url + "/prompt",
        data=json.dumps({"prompt": wf, "client_id": uuid.uuid4().hex}).encode(),
        headers={"Content-Type": "application/json"},
    )
    pid = json.load(urllib.request.urlopen(req, timeout=120))["prompt_id"]
    print("Генерирую тестовую картинку (30–120 секунд)...")

    img = None
    deadline = time.time() + 900
    while time.time() < deadline:
        try:
            h = json.load(urllib.request.urlopen(url + "/history/" + pid, timeout=60))
        except Exception:
            h = {}
        if pid in h:
            for node_out in h[pid].get("outputs", {}).values():
                for im in node_out.get("images", []):
                    img = im
                    break
            break
        time.sleep(4)

    if img:
        src = (url + "/view?filename=" + urllib.parse.quote(img["filename"])
               + "&subfolder=" + urllib.parse.quote(img.get("subfolder", ""))
               + "&type=" + img.get("type", "output"))
        data = urllib.request.urlopen(src, timeout=300).read()
        os.makedirs("/content/outputs", exist_ok=True)
        with open("/content/outputs/test.png", "wb") as f:
            f.write(data)
        from IPython.display import Image, display
        display(Image(data=data))
        print("✅ Тестовая картинка выше: сгенерирована и скачана ЧЕРЕЗ ТУННЕЛЬ — всё работает.")
    else:
        print("⚠️ Тестовая картинка не вышла, но сервер и туннель живы — проверь из Hermes.")

print()
print("=" * 56)
print("СКОПИРУЙ ЭТО И ПРИШЛИ В ЧАТ HERMES")
print("URL: " + url)
print("=" * 56)

---
## Дальше

- Этот ноутбук оставь запущенным — ссылка живёт, пока живёт сессия.
- Скажи Hermes «сгенерируй картинку: …» — он отправит запрос сюда.
- Сессия умерла? «Выполнить все» заново и пришли новую ссылку.
- Больше моделей/нод можно добавить через встроенный **ComfyUI-Manager** (меню в самом ComfyUI).